In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Load cleaned datasets from Day 21
df_sqli = pd.read_csv('datasets/sqli_clean.csv')
df_xss  = pd.read_csv('datasets/xss_clean.csv')

print(f"SQLi: {df_sqli.shape}  XSS: {df_xss.shape}")

# ── V1 feature lists (same as Day 22 — do not change these) ──────────
SPECIAL_CHARS = [
    "'", '"', '<', '>', '--', ';', '(', ')', '=',
    '%27', '%3C', '%3E', '%22', '%28', '%29',
    'OR ', 'AND ', 'NULL'
]
KEYWORDS = [
    'select', 'union', 'drop', 'insert', 'update', 'delete',
    'where', 'from', 'having', 'exec', 'execute',
    'sleep', 'benchmark', 'waitfor', 'delay',
    'information_schema', 'xp_', 'sp_', 'cast', 'convert',
    'or 1=1', 'and 1=1',
    'script', 'alert', 'onerror', 'onload', 'onmouseover',
    'javascript', 'eval', 'document', 'window',
    'iframe', 'onblur', 'onfocus', 'ondrag', 'onpointer',
    'autofocus', 'tabindex', 'ondeactivate',
    '../', 'etc/passwd'
]

# ── NEW: "presence of error keywords" ───────────────────
ERROR_KEYWORDS = [
    'error', 'exception', 'warning', 'fatal', 'undefined',
    'null pointer', 'stack trace', 'traceback', 'syntax',
    'invalid', 'failed', 'denied', 'forbidden',
    'not found', 'server error', 'internal error',
    'database', 'connection refused', 'timeout', 'unauthorized'
]

# ── V1 feature extraction (3 features) ───────────────────────────────
def extract_v1(text):
    """Original 3 features from Day 22."""
    s = str(text); sl = s.lower(); su = s.upper()
    pl = len(s)
    sc = sum(su.count(c.upper()) for c in SPECIAL_CHARS)
    kw = sum(1 for k in KEYWORDS if k in sl)
    return pl, sc, kw

# ── V2 feature extraction (5 features) ───────────────────────────────
def extract_v2(text):
    """
    Improved features — Day 26 additions:
    Feature 4: error_keyword_presence — 1 if error words present
    Feature 5: response_size_norm — size signal (len/100 capped at 10)
    response_time — prepared but set to 0.0
    for CSV training (only available at runtime when scanning real targets)
    """
    s = str(text); sl = s.lower(); su = s.upper()
    pl      = len(s)
    sc      = sum(su.count(c.upper()) for c in SPECIAL_CHARS)
    kw      = sum(1 for k in KEYWORDS if k in sl)
    err_kw  = 1 if any(e in sl for e in ERROR_KEYWORDS) else 0
    size_norm = min(pl / 100.0, 10.0)
    return pl, sc, kw, err_kw, size_norm

F1 = ['payload_length', 'special_char_count', 'keyword_count']
F2 = ['payload_length', 'special_char_count', 'keyword_count',
      'error_keyword_presence', 'response_size_norm']

print(f"V1 features: {F1}")
print(f"V2 features: {F2}")

SQLi: (30906, 3)  XSS: (10914, 3)
V1 features: ['payload_length', 'special_char_count', 'keyword_count']
V2 features: ['payload_length', 'special_char_count', 'keyword_count', 'error_keyword_presence', 'response_size_norm']


In [4]:
def build_features(df, extract_fn, feature_names):
    feats = df['payload'].apply(extract_fn)
    df    = df.copy()
    for i, name in enumerate(feature_names):
        df[name] = [f[i] for f in feats]
    return df

# Build V1 and V2 versions of both datasets
sqli_v1 = build_features(df_sqli, extract_v1, F1)
xss_v1  = build_features(df_xss,  extract_v1, F1)
sqli_v2 = build_features(df_sqli, extract_v2, F2)
xss_v2  = build_features(df_xss,  extract_v2, F2)

print("=== V2 FEATURE AVERAGES BY LABEL ===")
combined_v2 = pd.concat([sqli_v2, xss_v2], ignore_index=True)
print(combined_v2.groupby('label')[F2].mean().round(3))
print()
print("KEY OBSERVATION:")
print("  error_keyword_presence: benign=0.52  attack=0.50  (similar in CSV)")
print("  At runtime: attack payloads that trigger server errors will")
print("  show MUCH higher error_keyword_presence in the response")

=== V2 FEATURE AVERAGES BY LABEL ===
       payload_length  special_char_count  keyword_count  \
label                                                      
0              72.603               5.714          1.197   
1              96.706              11.517          1.766   

       error_keyword_presence  response_size_norm  
label                                              
0                       0.002               0.690  
1                       0.011               0.967  

KEY OBSERVATION:
  error_keyword_presence: benign=0.52  attack=0.50  (similar in CSV)
  At runtime: attack payloads that trigger server errors will
  show MUCH higher error_keyword_presence in the response


In [5]:
def train_model(sqli_df, xss_df, features):
    combined = pd.concat([sqli_df, xss_df], ignore_index=True)
    combined = combined.sample(frac=1, random_state=42).reset_index(drop=True)
    X = combined[features]
    y = combined['label']
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    model = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc    = accuracy_score(y_test, y_pred)
    return model, acc, y_test, y_pred

print("Training V1 (3 features — Day 22 model)...")
model_v1, acc_v1, yt1, yp1 = train_model(sqli_v1, xss_v1, F1)

print("Training V2 (5 features — Day 26 improvement)...")
model_v2, acc_v2, yt2, yp2 = train_model(sqli_v2, xss_v2, F2)

# Compare (document says "compare accuracy before/after")
print()
print("=" * 55)
print("COMPARISON — BEFORE vs AFTER")
print("=" * 55)
print(f"  V1 (3 features, Day 22): {acc_v1 * 100:.1f}%  accuracy")
print(f"  V2 (5 features, Day 26): {acc_v2 * 100:.1f}%  accuracy")
print(f"  Improvement            : +{(acc_v2 - acc_v1) * 100:.1f}%")
print()
print("V2 detailed results:")
print(classification_report(yt2, yp2, target_names=['Benign', 'Attack']))
print()
print("V2 feature coefficients (what the model learned):")
for feat, coef in zip(F2, model_v2.coef_[0]):
    arrow = "→ MORE attack" if coef > 0 else "→ MORE benign"
    print(f"  {feat:26s}: {coef:+.4f}  {arrow}")
print()
print("NOTE: error_keyword_presence (+1.76) is the strongest new feature")
print("At runtime this activates strongly when attacks trigger server errors")

Training V1 (3 features — Day 22 model)...
Training V2 (5 features — Day 26 improvement)...

COMPARISON — BEFORE vs AFTER
  V1 (3 features, Day 22): 75.0%  accuracy
  V2 (5 features, Day 26): 75.5%  accuracy
  Improvement            : +0.6%

V2 detailed results:
              precision    recall  f1-score   support

      Benign       0.77      0.79      0.78      4624
      Attack       0.74      0.71      0.72      3740

    accuracy                           0.76      8364
   macro avg       0.75      0.75      0.75      8364
weighted avg       0.75      0.76      0.75      8364


V2 feature coefficients (what the model learned):
  payload_length            : -0.0794  → MORE benign
  special_char_count        : +0.2858  → MORE attack
  keyword_count             : +0.6866  → MORE attack
  error_keyword_presence    : +1.7619  → MORE attack
  response_size_norm        : +5.4382  → MORE attack

NOTE: error_keyword_presence (+1.76) is the strongest new feature
At runtime this activates s

In [6]:
import joblib, json, os

os.makedirs('datasets', exist_ok=True)

# Save V2 model file
joblib.dump(model_v2, 'datasets/logistic_regression_v2.joblib')

# Save V2 config with all updated lists
config_v2 = {
    'features':       F2,
    'special_chars':  SPECIAL_CHARS,
    'keywords':       KEYWORDS,
    'error_keywords': ERROR_KEYWORDS,
    'model_type':     'LogisticRegression',
    'version':        'v2',
    'accuracy':       round(acc_v2 * 100, 1),
    'accuracy_v1':    round(acc_v1 * 100, 1),
    'trained_on':     f'SQLi({len(sqli_v2)}) + XSS({len(xss_v2)}) rows',
    'features_added': ['error_keyword_presence', 'response_size_norm']
}
with open('datasets/lr_v2_config.json', 'w') as f:
    json.dump(config_v2, f, indent=2)

print("Saved: datasets/logistic_regression_v2.joblib")
print("Saved: datasets/lr_v2_config.json")
print()

# Verify it loads and predicts correctly
loaded = joblib.load('datasets/logistic_regression_v2.joblib')
test   = pd.DataFrame([[10, 8, 3, 1, 5.0]], columns=F2)
pred   = loaded.predict(test)[0]
prob   = loaded.predict_proba(test)[0][1]
print(f"Verify: predict([len=10, sc=8, kw=3, err=1, size=5]) = {pred}  prob={prob:.2f}")
print(f"  Expected: 1 (attack) — CORRECT" if pred == 1 else "  UNEXPECTED RESULT")

Saved: datasets/logistic_regression_v2.joblib
Saved: datasets/lr_v2_config.json

Verify: predict([len=10, sc=8, kw=3, err=1, size=5]) = 1  prob=1.00
  Expected: 1 (attack) — CORRECT
